# 02 — Kaggle: offline execution of the harmonization engine

**Fire Harmonizer — MODIS–VIIRS physics-informed harmonization engine (NASA Space Apps 2026)**

This notebook is an **execution launcher**. It contains no harmonization, preprocessing,
statistical, physical, fusion, uncertainty or validation logic of its own: every computation
is performed by the project codebase shipped inside the input dataset published by
`01_colab_data_acquisition.ipynb`, through the codebase's own entry points:

| Stage | Codebase entry point | What the codebase does |
|---|---|---|
| Spatial subset (opt.) | Section 7b (`SPATIAL_SUBSET_MOD`) | contiguous ~25% of `2° × 2°` tiles across all 6 countries + 100% of all 6 Spotlights (whole H3-Res6 cells, all 26 years) |
| CPU prep | `scripts/run_env2_cpu_prep.py` | geometry, SZA, bow-tie, clustering, flare/SAMA hygiene, pyromes, 2012–2021 operator calibration, priors |
| Timing gate | `scripts/run_env2_gpu_solve.py --timing-gate-only` | readme Gate 4: one-chunk wall time + peak VRAM |
| GPU solve | `scripts/run_env2_gpu_solve.py` | multi-GPU float64 GMRF MAP + Takahashi + M=200 trajectories, conformal folds, Products 1–3 → DuckDB/Parquet |
| Validation | `scripts/run_env2_validate.py` | Chow/Pettitt, VIIRS-masked ablation re-solve, MCD64CMQ, residual slopes, spotlights → `diagnostics.json` |
| Hardware check | `tests/test_06_gpu_solver_math.py` | the codebase's own float64 / CUDA-kernel parity tests |

What this notebook does: locate and verify the package, resolve dependencies **offline** from
the bundled wheels, optionally build a cell-complete spatial subset for ~1–1.75h runs,
configure paths and GPUs, run the stages above with logging and resource monitoring, resume
completed stages, and collect/verify the outputs into `/kaggle/working/`.

**Internet is never used.** Nothing is downloaded from GitHub, NASA, PyPI or any API.

**Updating:** publish a new dataset version from the Colab notebook, then in Kaggle use the
input panel to attach/update to it and run all. This notebook needs no edits — it finds the
package by its `package_manifest.json`.

## 1. Configuration

Override any value with an environment variable `FH_<NAME>` if needed.
* `SPATIAL_SUBSET_MOD`: `4` keeps ~25% of contiguous `2° × 2°` tiles across all 6 countries plus 100% of all 6 Spotlights (~50 min – 1h50m total with full accuracy); `1` runs 100% of the dataset.

In [ ]:
import os, sys, json, re, time, shutil, hashlib, subprocess, platform, threading, zipfile, tarfile
import datetime as dt
from pathlib import Path


def _env(name, default, cast=str):
    v = os.environ.get("FH_" + name)
    if v is None:
        return default
    if cast is bool:
        return v.strip().lower() in ("1", "true", "yes", "on")
    if v.strip() == "" and default is None:
        return None
    return cast(v)


CONFIG = {
    "INPUT_ROOT": _env("INPUT_ROOT", "/kaggle/input"),
    "PACKAGE_DIR": _env("PACKAGE_DIR", ""),           # explicit package root (optional)
    "PACKAGE_ID": _env("PACKAGE_ID", ""),             # pick a specific package if several attached
    "WORK_ROOT": _env("WORK_ROOT", "/kaggle/tmp/fire_harmonizer"),
    "OUTPUT_ROOT": _env("OUTPUT_ROOT", "/kaggle/working"),
    "VERIFY_CHECKSUMS": _env("VERIFY_CHECKSUMS", True, bool),
    # spatial subsetting: 4 = ~25% contiguous 2x2 deg tiles + 100% of all 6 Spotlights; 1 = full 100%
    "SPATIAL_SUBSET_MOD": _env("SPATIAL_SUBSET_MOD", 4, int),
    # stages (all executed by the codebase)
    "RUN_HARDWARE_TESTS": _env("RUN_HARDWARE_TESTS", True, bool),   # tests/test_06
    "RUN_FULL_TEST_SUITE": _env("RUN_FULL_TEST_SUITE", False, bool),
    "STOP_ON_TEST_FAILURE": _env("STOP_ON_TEST_FAILURE", True, bool),
    "RUN_CPU_PREP": _env("RUN_CPU_PREP", True, bool),
    "RUN_TIMING_GATE": _env("RUN_TIMING_GATE", True, bool),
    "RUN_GPU_SOLVE": _env("RUN_GPU_SOLVE", True, bool),
    "RUN_VALIDATE": _env("RUN_VALIDATE", True, bool),
    "FORCE_RERUN": _env("FORCE_RERUN", False, bool),
    "REQUIRE_GPU": _env("REQUIRE_GPU", True, bool),
    # ---- GPU resource plan (section 8b) ----------------------------------------------
    # "auto"     : workers per GPU sized from RAM, GPU memory and the real input sizes
    # "safe"     : 1 worker per GPU          (lowest RAM, slowest)
    # "balanced" : 2 workers per GPU
    # "fast"     : 3 workers per GPU         (needs a lot of RAM)
    # "custom"   : exactly WORKERS_PER_GPU / BATCH_SIZE below
    "RESOURCE_PROFILE": _env("RESOURCE_PROFILE", "auto"),
    "WORKERS_PER_GPU": _env("WORKERS_PER_GPU", None, int),   # used by "custom"
    "BATCH_SIZE": _env("BATCH_SIZE", None, int),             # None = codebase default (1000)
    "MAX_SWEEPS": _env("MAX_SWEEPS", None, int),
    # out-of-memory protection: on an OOM kill the GPU stage is retried with fewer
    # workers (finished chunks are resumed, not recomputed) ...
    "AUTO_RETRY": _env("AUTO_RETRY", True, bool),
    # ... and, as a LAST resort, with smaller chunks (results then differ slightly,
    # because spatial smoothing couples cells only within a chunk; recorded in the report)
    "ALLOW_BATCH_REDUCTION": _env("ALLOW_BATCH_REDUCTION", True, bool),
    "RAM_SAFETY_FRACTION": _env("RAM_SAFETY_FRACTION", 0.80, float),
    "VRAM_SAFETY_FRACTION": _env("VRAM_SAFETY_FRACTION", 0.85, float),
    "RESUME_CHUNKS": _env("RESUME_CHUNKS", True, bool),
    # monitoring / housekeeping
    "MONITOR_INTERVAL_S": _env("MONITOR_INTERVAL_S", 15.0, float),
    "WORKING_QUOTA_GB": _env("WORKING_QUOTA_GB", 19.5, float),
    "DELETE_INTERMEDIATES": _env("DELETE_INTERMEDIATES", True, bool),
}

# Package contract shared with 01_colab_data_acquisition.ipynb
PACKAGE_SCHEMA_VERSION = 1
MANIFEST_NAME = "package_manifest.json"

OUTPUT_DIR = Path(CONFIG["OUTPUT_ROOT"]) / "output"        # codebase products
REPORT_DIR = Path(CONFIG["OUTPUT_ROOT"]) / "run_report"    # logs, metrics, provenance
WORK = Path(CONFIG["WORK_ROOT"])
try:
    WORK.mkdir(parents=True, exist_ok=True)
except OSError:
    WORK = Path("/tmp/fire_harmonizer")
    WORK.mkdir(parents=True, exist_ok=True)
for d in (OUTPUT_DIR, REPORT_DIR, REPORT_DIR / "logs"):
    d.mkdir(parents=True, exist_ok=True)
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(json.dumps(CONFIG, indent=2, default=str))
print("work:", WORK, "| outputs:", OUTPUT_DIR, "| report:", REPORT_DIR, "| run:", RUN_ID)

## 2. Locate the published package

Searches the attached inputs for `package_manifest.json` (any mount layout:
`/kaggle/input/<slug>/` or `/kaggle/input/datasets/<owner>/<slug>/`). If several packages are
attached, the most recently created one is used (or `PACKAGE_ID`). Package directories are
used in place when Kaggle has extracted them; if they arrive as `.zip`/`.tar` archives they
are extracted once into the work area.

In [ ]:
def find_manifests(root, max_depth=6):
    root = Path(root)
    found = []
    if not root.exists():
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth >= max_depth:
            dirnames[:] = []
        if MANIFEST_NAME in filenames:
            found.append(Path(dirpath) / MANIFEST_NAME)
            dirnames[:] = []          # never descend into a package
    return found


if CONFIG["PACKAGE_DIR"]:
    candidates = [Path(CONFIG["PACKAGE_DIR"]) / MANIFEST_NAME]
else:
    candidates = find_manifests(CONFIG["INPUT_ROOT"])
if not candidates:
    raise FileNotFoundError(
        f"no {MANIFEST_NAME} under {CONFIG['INPUT_ROOT']}: attach the dataset published by "
        "01_colab_data_acquisition.ipynb (Add Input -> Your Datasets)")
loaded = []
for c in candidates:
    m = json.loads(c.read_text())
    if m.get("schema_version") != PACKAGE_SCHEMA_VERSION:
        print(f"skipping {c}: schema {m.get('schema_version')} != {PACKAGE_SCHEMA_VERSION}")
        continue
    loaded.append((m, c.parent))
if CONFIG["PACKAGE_ID"]:
    loaded = [x for x in loaded if x[0]["package_id"] == CONFIG["PACKAGE_ID"]]
if not loaded:
    raise RuntimeError("no compatible package found")
MANIFEST, PKG_ROOT = sorted(loaded, key=lambda x: x[0]["created_utc"])[-1]
PACKAGE_ID = MANIFEST["package_id"]
print(f"package {PACKAGE_ID}\n  at {PKG_ROOT}\n  created {MANIFEST['created_utc']}\n"
      f"  codebase {MANIFEST['codebase']['repo_url']} @ {MANIFEST['codebase']['commit']}\n"
      f"  files {MANIFEST['totals']['files']}, {MANIFEST['totals']['bytes'] / 2**30:.2f} GiB")
if len(loaded) > 1:
    print("other attached packages:", [m["package_id"] for m, _ in loaded[:-1]])


def resolve_component(name):
    # Return a directory holding package component `name` (in place or extracted).
    direct = PKG_ROOT / name
    if direct.is_dir():
        return direct
    for ext in (".zip", ".tar", ".tar.gz", ".tgz"):
        arc = PKG_ROOT / (name + ext)
        if arc.is_file():
            dest = WORK / "extracted" / PACKAGE_ID / name
            done = dest.parent / f".{name}.extracted"
            if not done.exists():
                if dest.exists():
                    shutil.rmtree(dest)
                dest.mkdir(parents=True)
                print(f"extracting {arc.name} -> {dest}")
                if ext == ".zip":
                    with zipfile.ZipFile(arc) as zf:
                        zf.extractall(dest)
                else:
                    with tarfile.open(arc) as tf:
                        tf.extractall(dest, filter="data")
                # archives made from a directory may contain it as a single top folder
                kids = list(dest.iterdir())
                if len(kids) == 1 and kids[0].is_dir() and kids[0].name == name:
                    for k in list(kids[0].iterdir()):
                        shutil.move(str(k), dest / k.name)
                    kids[0].rmdir()
                done.write_text(PACKAGE_ID)
            return dest
    raise FileNotFoundError(f"package component '{name}' not found under {PKG_ROOT}")


COMPONENTS = {key: resolve_component(name) for key, name in MANIFEST["layout"].items()}
for k, v in COMPONENTS.items():
    print(f"  {k:<12} -> {v}")

## 3. Verify package integrity (SHA-256 against the manifest)

Every file listed in the manifest must exist with the recorded size and checksum; missing or
corrupted files stop the run. Unexpected extra files are reported.

In [ ]:
def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


name_to_key = {v: k for k, v in MANIFEST["layout"].items()}


def local_path(rel):
    head, _, rest = rel.partition("/")
    if head in name_to_key and rest:
        return COMPONENTS[name_to_key[head]] / rest
    return PKG_ROOT / rel


t0 = time.time()
missing, bad_size, bad_hash = [], [], []
for rel, info in MANIFEST["files"].items():
    p = local_path(rel)
    if not p.is_file():
        missing.append(rel)
        continue
    if p.stat().st_size != info["bytes"]:
        bad_size.append(rel)
        continue
    if CONFIG["VERIFY_CHECKSUMS"] and sha256_file(p) != info["sha256"]:
        bad_hash.append(rel)
listed = {local_path(r).resolve() for r in MANIFEST["files"]}
extra = [str(p) for key in COMPONENTS for p in COMPONENTS[key].rglob("*")
         if p.is_file() and p.resolve() not in listed and not p.name.startswith(".")]
print(f"checked {len(MANIFEST['files'])} files in {time.time() - t0:.0f}s "
      f"(checksums {'on' if CONFIG['VERIFY_CHECKSUMS'] else 'off'})")
if extra:
    print(f"note: {len(extra)} files not in the manifest, e.g. {extra[:3]}")
if missing or bad_size or bad_hash:
    raise RuntimeError(f"package integrity failure: missing={missing[:10]} "
                       f"size={bad_size[:10]} sha256={bad_hash[:10]}")
print("package integrity OK")

## 4. Prepare the codebase and check compatibility

The codebase snapshot is copied to a writable work directory (Python caches, compiled CUDA
extensions). Compatibility checks: the snapshot's `REVISION.json` matches the manifest
commit, the codebase's entry points exist, and the running interpreter matches the wheels'
target Python.

In [ ]:
CODE_SRC = COMPONENTS["codebase"]
CODE = WORK / "codebase" / PACKAGE_ID
if not (CODE / ".copied").exists():
    if CODE.exists():
        shutil.rmtree(CODE)
    shutil.copytree(CODE_SRC, CODE)
    (CODE / ".copied").write_text(PACKAGE_ID)

rev = json.loads((CODE / "REVISION.json").read_text())
assert rev["commit"] == MANIFEST["codebase"]["commit"], "codebase snapshot != manifest commit"

ENTRYPOINTS = {
    "cpu_prep": "scripts/run_env2_cpu_prep.py",
    "gpu_solve": "scripts/run_env2_gpu_solve.py",
    "validate": "scripts/run_env2_validate.py",
    "hardware_tests": "tests/test_06_gpu_solver_math.py",
}
absent = [p for p in ENTRYPOINTS.values() if not (CODE / p).exists()]
if absent:
    raise RuntimeError(f"codebase snapshot lacks entry points {absent}")

py = f"{sys.version_info.major}.{sys.version_info.minor}"
target = MANIFEST.get("target_env2_python")
print(f"interpreter {platform.python_version()} | wheels built for {target} | commit {rev['commit'][:12]}")
if target and target != py:
    print(f"WARNING: wheels target Python {target} but this kernel runs {py}; "
          f"binary wheels (h3) may not install. Re-publish with TARGET_ENV2_PYTHON={py}.")

## 5. Resolve dependencies offline

Reads the codebase's `engine`, `webapp` and `dev` dependency groups from `pyproject.toml`,
keeps every package already provided by the Kaggle image, and installs only missing ones
from the bundled wheels with `pip --no-index` (no network).

In [ ]:
import importlib, importlib.metadata as md_

try:
    import tomllib
except ImportError:
    import tomli as tomllib
pyproject = tomllib.loads((CODE / "pyproject.toml").read_text())
groups = pyproject["project"]["optional-dependencies"]
reqs = sorted({re.split(r"[<>=\[ ;]", r)[0] for g in ("engine", "webapp", "dev")
               for r in groups.get(g, [])})
IMPORT_NAMES = {"scikit-learn": "sklearn", "pyyaml": "yaml"}


def installed_version(pkg):
    try:
        return md_.version(pkg)
    except md_.PackageNotFoundError:
        return None


missing_pkgs = [r for r in reqs if installed_version(r) is None]
WHEELS = COMPONENTS.get("wheels")
if missing_pkgs:
    print("installing offline from bundled wheels:", missing_pkgs)
    if WHEELS is None:
        raise RuntimeError(f"missing {missing_pkgs} and the package has no wheel directory")
    p = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                        "--find-links", str(WHEELS), *missing_pkgs],
                       capture_output=True, text=True)
    print(p.stdout[-1500:], p.stderr[-1500:])
    if p.returncode != 0:
        raise RuntimeError(f"offline install failed for {missing_pkgs}")
    importlib.invalidate_caches()

DEP_VERSIONS = {}
for r in reqs:
    v = installed_version(r)
    if v is None:
        raise RuntimeError(f"dependency {r} unavailable after offline install")
    importlib.import_module(IMPORT_NAMES.get(r, r))
    DEP_VERSIONS[r] = v
print(json.dumps(DEP_VERSIONS, indent=1))

## 6. Hardware inventory

Kaggle's 4× L4 are **four separate 22–24 GiB devices**, not one 96 GB pool. The codebase
partitions active cells into chunks sized for one GPU (`batch_size = 1000` cells,
< 6 GiB each) and spreads chunks over all devices with `torch.multiprocessing`
(`--workers-per-gpu` can place more than one chunk worker per GPU).

In [ ]:
import psutil
import torch

HW = {"cpu_logical": psutil.cpu_count(), "ram_total_gib": round(psutil.virtual_memory().total / 2**30, 1),
      "torch": torch.__version__, "cuda_available": torch.cuda.is_available(), "gpus": []}
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        HW["gpus"].append({"index": i, "name": prop.name,
                           "vram_gib": round(prop.total_memory / 2**30, 2),
                           "capability": f"{prop.major}.{prop.minor}"})
try:
    from torch.utils.cpp_extension import CUDA_HOME
    HW["cuda_home"] = CUDA_HOME
    HW["nvcc"] = bool(CUDA_HOME) and Path(CUDA_HOME, "bin", "nvcc").exists()
except Exception as exc:
    HW["cuda_home"], HW["nvcc"] = None, False
for path in (CONFIG["OUTPUT_ROOT"], str(WORK)):
    du = shutil.disk_usage(path)
    HW[f"disk_free_gib[{path}]"] = round(du.free / 2**30, 1)
print(json.dumps(HW, indent=1))
if not HW["gpus"]:
    msg = "no CUDA GPU visible: the codebase will fall back to CPU (very slow on real data)"
    if CONFIG["REQUIRE_GPU"]:
        raise RuntimeError(msg + " -- enable GPU accelerators or set FH_REQUIRE_GPU=0")
    print("WARNING:", msg)

## 7. Configure the codebase environment

The codebase's single path resolver (`src/config/paths.py`, `DataPaths.from_env`) honours
`FIRE_HARMONIZER_STAGED_DIR`, `FIRE_HARMONIZER_WHEELS_DIR` and `FIRE_HARMONIZER_OUTPUT_DIR`.
The notebook sets them and asks the codebase itself how it resolved every path.

In [ ]:
RUN_ENV = dict(os.environ)
RUN_ENV.update({
    "FIRE_HARMONIZER_STAGED_DIR": str(COMPONENTS["staged_data"]),
    "FIRE_HARMONIZER_WHEELS_DIR": str(COMPONENTS.get("wheels", "")),
    "FIRE_HARMONIZER_OUTPUT_DIR": str(OUTPUT_DIR),
    "PYTHONPATH": str(CODE) + os.pathsep + os.environ.get("PYTHONPATH", ""),
    "TORCH_EXTENSIONS_DIR": str(WORK / "torch_extensions"),
    "PYTHONUNBUFFERED": "1",
})
RESUME_DIR = WORK / "solve_resume"           # finished GMRF chunks (resumable solves)
if CONFIG["RESUME_CHUNKS"]:
    RESUME_DIR.mkdir(parents=True, exist_ok=True)
    RUN_ENV["FIRE_HARMONIZER_RESUME_DIR"] = str(RESUME_DIR)
_probe = subprocess.run([sys.executable, "-c", (
    "import json; from src.config.paths import DataPaths; p = DataPaths.from_env(); "
    "print(json.dumps({k: str(v) for k, v in vars(p).items()}))")],
    cwd=CODE, env=RUN_ENV, capture_output=True, text=True)
if _probe.returncode != 0:
    raise RuntimeError(_probe.stderr)
CODE_PATHS = json.loads(_probe.stdout.strip().splitlines()[-1])
for k in ("active_fires_dir", "cloud_parquet", "country_polygons_parquet",
          "burned_area_parquet", "duckdb_path", "diagnostics_json"):
    print(f"  {k:<26} {CODE_PATHS[k]}")
assert Path(CODE_PATHS["active_fires_dir"]).is_dir(), "codebase cannot see active_fires/"
assert CODE_PATHS["duckdb_path"].startswith(str(OUTPUT_DIR)), "outputs would not land in /kaggle/working"

## 7b. Spatial subsetting (contiguous `2° × 2°` tiles + all 6 Spotlights)

When `SPATIAL_SUBSET_MOD > 1` (default `4` ≈ 25%), builds a cell-complete subset under
`WORK / "staged_subset"` keeping:
* ~`1 / SPATIAL_SUBSET_MOD` of contiguous `2° × 2°` tiles across all 6 countries (preserving 1-ring spatial neighbors, all biomes, and `2012–2021` calibration diversity),
* **100%** of all 6 Spotlight bounding boxes,
* **100%** of all 26 years (`2000–2026`) and **every detection** of each kept H3-Res6 hexagon (boundary detections snap by exact H3-Res6 cell center so no hexagon is ever split).

Set `SPATIAL_SUBSET_MOD = 1` in Section 1 to run the full 100% dataset.

In [ ]:
if CONFIG["SPATIAL_SUBSET_MOD"] > 1:
    import numpy as np
    import h3
    import pyarrow as pa
    import pyarrow.parquet as pq

    SUBSET_MOD = int(CONFIG["SPATIAL_SUBSET_MOD"])
    SUBSET_DIR = WORK / "staged_subset"
    src_root = COMPONENTS["staged_data"]
    src_af, dst_af = src_root / "active_fires", SUBSET_DIR / "active_fires"
    marker = SUBSET_DIR / ".done"
    tag_prefix = f"{PACKAGE_ID}:mod={SUBSET_MOD}:"

    if not (marker.exists() and marker.read_text().startswith(tag_prefix)):
        if SUBSET_DIR.exists():
            shutil.rmtree(SUBSET_DIR)
        SUBSET_DIR.mkdir(parents=True)
        for item in src_root.iterdir():
            if item.name != "active_fires":
                dst = SUBSET_DIR / item.name
                try:
                    os.symlink(item, dst)
                except OSError:
                    shutil.copytree(item, dst) if item.is_dir() else shutil.copy2(item, dst)

        sys.path.insert(0, str(CODE))
        try:
            from src.config import settings as _S
            SPOTLIGHTS = [sp.bbox for sp in _S.SPOTLIGHT_PRESETS]
        finally:
            sys.path.remove(str(CODE))

        TILE_DEG, PAD = 2.0, 0.25  # PAD > max H3-Res6 circumradius even at 72 deg N (Alaska)
        _to_cell = h3.latlng_to_cell if hasattr(h3, "latlng_to_cell") else h3.geo_to_h3
        _to_ll = h3.cell_to_latlng if hasattr(h3, "cell_to_latlng") else h3.h3_to_geo

        def _tile_selected(lat, lon):
            gi = np.floor(lat / TILE_DEG).astype(np.int64)
            gj = np.floor(lon / TILE_DEG).astype(np.int64)
            spot = np.zeros(len(lat), dtype=bool)
            for w, s, e, n in SPOTLIGHTS:
                spot |= (lon >= w - PAD) & (lon <= e + PAD) & (lat >= s - PAD) & (lat <= n + PAD)
            h = ((gi * np.int64(73856093)) ^ (gj * np.int64(19349663))) % np.int64(SUBSET_MOD)
            return spot | (h == 0)

        def _keep_whole_hexes(lat, lon):
            valid = np.isfinite(lat) & np.isfinite(lon)
            keep = np.zeros(len(lat), dtype=bool)
            if not valid.any():
                return keep
            la, lo = lat[valid], lon[valid]
            s0 = _tile_selected(la, lo)
            s1 = _tile_selected(la - PAD, lo - PAD)
            s2 = _tile_selected(la - PAD, lo + PAD)
            s3 = _tile_selected(la + PAD, lo - PAD)
            s4 = _tile_selected(la + PAD, lo + PAD)
            decided = (s0 == s1) & (s0 == s2) & (s0 == s3) & (s0 == s4)
            sub = s0.copy()
            mixed = np.flatnonzero(~decided)
            if len(mixed):
                cache = {}
                clat = np.empty(len(mixed), dtype=np.float64)
                clon = np.empty(len(mixed), dtype=np.float64)
                for k, i in enumerate(mixed):
                    c = _to_cell(float(la[i]), float(lo[i]), 6)
                    ll = cache.get(c)
                    if ll is None:
                        ll = _to_ll(c)
                        cache[c] = ll
                    clat[k], clon[k] = ll
                sub[mixed] = _tile_selected(clat, clon)
            keep[valid] = sub
            return keep

        files = sorted(src_af.rglob("*.parquet"))
        n_in = n_out = 0
        t0 = time.time()
        for fi, fp in enumerate(files):
            t = pq.read_table(fp, partitioning=None)
            n_in += t.num_rows
            if t.num_rows == 0:
                continue
            lat = np.asarray(t.column("latitude").combine_chunks().to_numpy(zero_copy_only=False), dtype=np.float64)
            lon = np.asarray(t.column("longitude").combine_chunks().to_numpy(zero_copy_only=False), dtype=np.float64)
            mask = _keep_whole_hexes(lat, lon)
            k = int(mask.sum())
            if k > 0:
                out_fp = dst_af / fp.relative_to(src_af)
                out_fp.parent.mkdir(parents=True, exist_ok=True)
                pq.write_table(t.filter(pa.array(mask)), out_fp, compression="zstd")
                n_out += k
            if (fi + 1) % 300 == 0 or fi + 1 == len(files):
                print(f"subset {fi + 1}/{len(files)} files: kept {n_out:,} / {n_in:,} "
                      f"({100 * n_out / max(n_in, 1):.1f}%), {time.time() - t0:.0f}s")
        marker.write_text(f"{tag_prefix}{n_out}/{n_in}")

    SUBSET_INFO = marker.read_text()
    RUN_ENV["FIRE_HARMONIZER_STAGED_DIR"] = str(SUBSET_DIR)
    _probe = subprocess.run([sys.executable, "-c", (
        "import json; from src.config.paths import DataPaths; p = DataPaths.from_env(); "
        "print(json.dumps({k: str(v) for k, v in vars(p).items()}))")],
        cwd=CODE, env=RUN_ENV, capture_output=True, text=True, check=True)
    CODE_PATHS = json.loads(_probe.stdout.strip().splitlines()[-1])
    print(f"spatial subset active ({SUBSET_INFO}) -> {CODE_PATHS['active_fires_dir']}")
else:
    SUBSET_INFO = "full (SPATIAL_SUBSET_MOD <= 1)"
    print("SPATIAL_SUBSET_MOD <= 1: using full staged dataset")

## 8. Execution monitor and stage runner

* `ResourceMonitor` samples CPU %, RAM and, per GPU, utilisation and memory (`nvidia-smi`)
  every `MONITOR_INTERVAL_S` seconds into `run_report/resources.csv` (GPU work runs in the
  codebase's worker subprocesses, so device-level sampling is used).
* `run_stage` runs one codebase entry point, streams its log, records timing and exit code,
  and writes a completion marker keyed by package, commit, subset setting, arguments and
  upstream markers — re-running the notebook skips stages that already completed with
  identical inputs (`FORCE_RERUN = True` to redo).
* **Resource plan (8b).** `RESOURCE_PROFILE = "auto"` sizes the number of chunk workers per
  GPU from total RAM, GPU memory and the real input sizes (re-planned with the GPU memory the
  timing gate measures). If a GPU stage is killed for lack of memory it is retried down a
  ladder — fewer workers per GPU, then (last resort) smaller chunks — and every finished chunk
  is resumed instead of recomputed. Real errors (not memory) stop immediately;
  per-chunk MAP states are checkpointed by the codebase under `output/state_checkpoint.pt.chunks/`.

In [ ]:
class ResourceMonitor(threading.Thread):
    def __init__(self, path, interval):
        super().__init__(daemon=True)
        self.path, self.interval = Path(path), interval
        self._halt = threading.Event()
        self.peaks = {"ram_gib": 0.0, "gpu_mem_gib": {}}
        new = not self.path.exists()
        self.fh = open(self.path, "a")
        if new:
            self.fh.write("utc,stage,cpu_pct,ram_used_gib,gpu,gpu_util_pct,gpu_mem_used_gib\n")
        self.stage = "idle"

    def sample(self):
        ts = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
        cpu = psutil.cpu_percent(interval=None)
        ram = psutil.virtual_memory().used / 2**30
        self.peaks["ram_gib"] = max(self.peaks["ram_gib"], ram)
        rows = []
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used",
                                  "--format=csv,noheader,nounits"], capture_output=True,
                                 text=True, timeout=10).stdout
            for line in out.strip().splitlines():
                i, util, mem = [x.strip() for x in line.split(",")]
                g = float(mem) / 1024
                self.peaks["gpu_mem_gib"][i] = max(self.peaks["gpu_mem_gib"].get(i, 0.0), g)
                rows.append((i, util, f"{g:.3f}"))
        except (FileNotFoundError, subprocess.TimeoutExpired, ValueError):
            pass
        for i, util, g in rows or [("", "", "")]:
            self.fh.write(f"{ts},{self.stage},{cpu},{ram:.3f},{i},{util},{g}\n")
        self.fh.flush()

    def run(self):
        psutil.cpu_percent(interval=None)
        while not self._halt.wait(self.interval):
            self.sample()

    def stop(self):
        self._halt.set()
        self.sample()
        self.fh.close()


MONITOR = ResourceMonitor(REPORT_DIR / "resources.csv", CONFIG["MONITOR_INTERVAL_S"])
MONITOR.start()

STATE_DIR = OUTPUT_DIR / "_run_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
STAGES = {}


def _fingerprint(name, args, upstream):
    parts = [PACKAGE_ID, MANIFEST["codebase"]["commit"],
             f"subset_mod={CONFIG['SPATIAL_SUBSET_MOD']}", name, json.dumps(args)]
    for u in upstream:
        mp = STATE_DIR / f"{u}.json"
        parts.append(mp.read_text() if mp.exists() else "missing")
    return hashlib.sha256("|".join(parts).encode()).hexdigest()


class StageFailed(RuntimeError):
    returncode, tail = None, ""


def run_stage(name, args, expected_outputs=(), upstream=(), cwd=None):
    marker = STATE_DIR / f"{name}.json"
    fp = _fingerprint(name, args, upstream)
    if not CONFIG["FORCE_RERUN"] and marker.exists():
        prev = json.loads(marker.read_text())
        if prev.get("fingerprint") == fp and all(Path(p).exists() for p in expected_outputs):
            print(f"[{name}] already completed for this package/commit/args -> skipped "
                  f"(finished {prev['finished_utc']})")
            STAGES[name] = {**prev, "skipped": True}
            return STAGES[name]
    log_path = REPORT_DIR / "logs" / f"{name}.log"
    cmd = [sys.executable, *args]
    print(f"[{name}] $ {' '.join(cmd)}\n[{name}] log: {log_path}")
    MONITOR.stage = name
    t0 = time.time()
    started = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    tail = []
    with open(log_path, "a") as log:
        log.write(f"\n===== {started} {' '.join(cmd)}\n")
        proc = subprocess.Popen(cmd, cwd=cwd or CODE, env=RUN_ENV, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            tail = (tail + [line])[-80:]
            print(line, end="")
        rc = proc.wait()
    MONITOR.stage = "idle"
    rec = {"stage": name, "cmd": cmd, "returncode": rc, "seconds": round(time.time() - t0, 1),
           "started_utc": started,
           "finished_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
           "log": str(log_path), "fingerprint": fp, "skipped": False}
    STAGES[name] = rec
    if rc != 0:
        marker.unlink(missing_ok=True)
        err = StageFailed(f"[{name}] failed with exit code {rc} after {rec['seconds']}s; "
                          "last log lines:\n" + "".join(tail[-40:]))
        err.returncode, err.tail = rc, "".join(tail)
        raise err
    absent = [p for p in expected_outputs if not Path(p).exists()]
    if absent:
        raise RuntimeError(f"[{name}] finished but expected outputs are missing: {absent}")
    marker.write_text(json.dumps(rec, indent=2))
    print(f"[{name}] OK in {rec['seconds']}s")
    return rec


# ---------------------------------------------------------------------------------
# 8b. GPU resource plan: how many chunk workers per GPU, which chunk size, and the
#     fallback ladder used if a GPU stage is killed for lack of memory.
# ---------------------------------------------------------------------------------
GiB = 2 ** 30
OOM_PATTERN = re.compile(
    r"out of memory|OutOfMemoryError|MemoryError|Cannot allocate memory|Killed|SIGKILL|"
    r"signal 9|ProcessExitedException|No space left on device|std::bad_alloc", re.I)
PROFILE_WORKERS = {"safe": 1, "balanced": 2, "fast": 3}
DEFAULT_BATCH = 1000                     # codebase default (settings.BATCH_SIZE)


def is_resource_failure(exc):
    """OOM kill / CUDA OOM / disk full -> retry with less; anything else is a real
    error and is NOT retried (retrying a bug only wastes GPU quota)."""
    rc = getattr(exc, "returncode", None)
    return rc in (-9, 137, -6, 134) or bool(OOM_PATTERN.search(getattr(exc, "tail", "")))


def _parquet_mem_gib(path):
    import pyarrow.parquet as pq
    p = Path(path)
    files = sorted(p.rglob("*.parquet")) if p.is_dir() else ([p] if p.exists() else [])
    tot = 0
    for f in files:
        md = pq.ParquetFile(f).metadata
        tot += sum(md.row_group(i).total_byte_size for i in range(md.num_row_groups))
    return tot / GiB


def hardware_inventory():
    gpus = []
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True,
                             text=True, timeout=20).stdout
        for line in out.strip().splitlines():
            i, name, mem = [x.strip() for x in line.split(",")]
            gpus.append({"index": int(i), "name": name, "vram_gib": float(mem) / 1024})
    except (FileNotFoundError, subprocess.TimeoutExpired, ValueError):
        pass
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage(WORK)
    return {"gpus": gpus, "ram_total_gib": vm.total / GiB, "ram_avail_gib": vm.available / GiB,
            "cpu_cores": os.cpu_count(), "work_disk_free_gib": disk.free / GiB}


def plan_resources(measured_vram_gib=None):
    """Return (plan dict, ladder [(workers_per_gpu, batch_size, note), ...])."""
    import pyarrow.parquet as pq
    hw = hardware_inventory()
    n_gpu = max(1, len(hw["gpus"]))
    vram = min((g["vram_gib"] for g in hw["gpus"]), default=0.0)
    batch = CONFIG["BATCH_SIZE"] or DEFAULT_BATCH
    # data-driven memory model (all GiB)
    shared = (1.3 * (_parquet_mem_gib(CODE_PATHS["pass_calibrated_parquet"])
                     + _parquet_mem_gib(CODE_PATHS["cloud_parquet"])
                     + _parquet_mem_gib(CODE_PATHS["pyromes_parquet"]))
              + 1.2 * sum(Path(CODE_PATHS[k]).stat().st_size / GiB
                          for k in ("priors_pkl", "calibration_pkl")
                          if Path(CODE_PATHS[k]).exists()))
    try:
        n_cells = len(set(pq.read_table(CODE_PATHS["pass_calibrated_parquet"],
                                        columns=["h3_index"]).column(0).to_pylist()))
    except Exception:
        n_cells = 0
    chunk_ram = 6.0 * batch / 1000            # dense (cells x 9510 days x 5 sats x 2) arrays
    monthly = n_cells * 320 * 250 / GiB       # finalized monthly frame in the parent
    parent = 2.0 * shared + 2.0 * monthly + 4.0
    per_worker = shared + chunk_ram
    vram_per_worker = (measured_vram_gib or 4.5 * batch / 1000) + 0.5
    budget = CONFIG["RAM_SAFETY_FRACTION"] * hw["ram_total_gib"]
    w_ram = int(max(0.0, budget - parent) // (n_gpu * per_worker)) if per_worker else 1
    w_vram = int(CONFIG["VRAM_SAFETY_FRACTION"] * vram // vram_per_worker) if vram else 1
    w_auto = max(1, min(4, w_ram, w_vram))
    prof = CONFIG["RESOURCE_PROFILE"].lower()
    if prof == "custom":
        start = CONFIG["WORKERS_PER_GPU"] or 1
    elif prof in PROFILE_WORKERS:
        start = PROFILE_WORKERS[prof]
    else:
        prof, start = "auto", w_auto
    ladder = [(w, batch, "planned" if w == start else "fallback: fewer workers")
              for w in range(start, 0, -1)]
    if CONFIG["ALLOW_BATCH_REDUCTION"]:
        for b in (batch // 2, batch // 4):
            if b >= 100:
                ladder.append((1, b, "fallback: smaller chunks (results differ slightly)"))
    if not CONFIG["AUTO_RETRY"]:
        ladder = ladder[:1]
    est_ram = parent + n_gpu * start * per_worker
    plan = {"profile": prof, "hardware": hw, "n_active_cells": n_cells,
            "est_shared_inputs_gib": round(shared, 1), "est_parent_gib": round(parent, 1),
            "est_per_worker_gib": round(per_worker, 1),
            "est_ram_at_start_gib": round(est_ram, 1),
            "auto_limit_by_ram": w_ram, "auto_limit_by_vram": w_vram,
            "measured_vram_per_chunk_gib": measured_vram_gib,
            "ladder": [{"workers_per_gpu": w, "batch_size": b, "note": n}
                       for w, b, n in ladder]}
    if est_ram > hw["ram_total_gib"]:
        print(f"WARNING: profile {prof!r} needs ~{est_ram:.0f} GiB RAM but the machine has "
              f"{hw['ram_total_gib']:.0f} GiB; the fallback ladder will step down if needed")
    need_disk = (n_cells / max(1, batch)) * 0.12 * 1.5
    if CONFIG["RESUME_CHUNKS"] and hw["work_disk_free_gib"] < need_disk:
        print(f"WARNING: ~{need_disk:.0f} GiB needed for resumable chunks but only "
              f"{hw['work_disk_free_gib']:.0f} GiB free in {WORK}: chunk resume disabled")
        RUN_ENV.pop("FIRE_HARMONIZER_RESUME_DIR", None)
    return plan, ladder


def engine_args(workers, batch):
    a = ["--workers-per-gpu", str(workers)]
    if batch != DEFAULT_BATCH or CONFIG["BATCH_SIZE"]:
        a += ["--batch-size", str(batch)]
    return a


RESOURCE_LOG = []


def run_engine_stage(name, base_args, ladder, expected_outputs=(), upstream=(),
                     extra_args=()):
    """run_stage over the fallback ladder: an out-of-memory kill moves to the next
    rung (finished chunks are resumed); any other failure stops immediately."""
    last = None
    for k, (w, b, note) in enumerate(ladder):
        args = [*base_args, *engine_args(w, b), *extra_args]
        print(f"[{name}] attempt {k + 1}/{len(ladder)}: {w} worker(s)/GPU, "
              f"chunk {b} cells ({note})")
        try:
            rec = run_stage(name, args, expected_outputs, upstream)
            RESOURCE_LOG.append({"stage": name, "workers_per_gpu": w, "batch_size": b,
                                 "note": note, "result": "ok"})
            return rec
        except StageFailed as exc:
            last = exc
            oom = is_resource_failure(exc)
            RESOURCE_LOG.append({"stage": name, "workers_per_gpu": w, "batch_size": b,
                                 "note": note, "returncode": exc.returncode,
                                 "result": "out-of-memory" if oom else "error"})
            if not oom:
                raise
            print(f"[{name}] ran out of memory with {w} worker(s)/GPU, chunk {b}; "
                  "stepping down (finished chunks are kept)")
            time.sleep(10)            # let the killed workers release GPU memory
    raise RuntimeError(f"[{name}] failed on every rung of the resource ladder") from last


print("resource profile:", CONFIG["RESOURCE_PROFILE"], "| auto retry:", CONFIG["AUTO_RETRY"],
      "| chunk resume:", "FIRE_HARMONIZER_RESUME_DIR" in RUN_ENV)
print("hardware:", json.dumps(hardware_inventory(), default=str))

## 9. Codebase hardware tests (float64 solver math + CUDA kernels on these GPUs)

Runs the codebase's own `tests/test_06_gpu_solver_math.py`: Thomas/Takahashi/Cholesky
reference math and — on GPU hosts — the `load_inline` CUDA kernel build, per-device
residency and bit-parity tests. `RUN_FULL_TEST_SUITE` runs all ten suites (≈ 15–20 min,
synthetic digital twin included).

In [ ]:
TESTS = {}


def run_tests(name, target):
    cmd = ["-m", "pytest", target, "-q", "-rfs", "-p", "no:cacheprovider", "-o", "addopts="]
    try:
        rec = run_stage(name, cmd)
        TESTS[name] = {"passed": True, **rec}
    except RuntimeError as exc:
        TESTS[name] = {"passed": False, "error": str(exc)[-2000:]}
        if CONFIG["STOP_ON_TEST_FAILURE"]:
            raise
        print("continuing despite test failure (STOP_ON_TEST_FAILURE=False)")
    log = Path(REPORT_DIR / "logs" / f"{name}.log").read_text()
    summary = [l for l in log.splitlines() if re.search(r"\d+ (passed|failed|skipped)", l)]
    TESTS[name]["summary"] = summary[-1] if summary else ""
    print(f"[{name}] {TESTS[name]['summary']}")


if CONFIG["RUN_HARDWARE_TESTS"]:
    run_tests("tests_hardware", ENTRYPOINTS["hardware_tests"])
if CONFIG["RUN_FULL_TEST_SUITE"]:
    run_tests("tests_full", "tests")

## 10. Stage 1 — CPU preparation (codebase `run_env2_cpu_prep.py`)

In [ ]:
CPU_PREP_OUTPUTS = [CODE_PATHS[k] for k in (
    "provenance_base_parquet", "pass_obs_parquet", "pyromes_parquet", "calibration_pkl",
    "pass_calibrated_parquet", "priors_pkl")]
if CONFIG["RUN_CPU_PREP"]:
    run_stage("cpu_prep", [ENTRYPOINTS["cpu_prep"]], CPU_PREP_OUTPUTS)
else:
    print("skipped by configuration")

## 11. Stage 2 — timing gate (readme Gate 4) and Stage 3 — GPU solve

The timing gate benchmarks one chunk (wall time, peak VRAM, projected GPU-hours) before the
full run. The full solve then runs the main reconstruction, the two conformal hold-out folds,
and exports `fire_harmonized.duckdb` + the three Parquet products.

In [ ]:
PLAN, LADDER = plan_resources()
gate_batch = LADDER[0][1]
if CONFIG["RUN_TIMING_GATE"]:
    run_engine_stage("timing_gate", [ENTRYPOINTS["gpu_solve"], "--timing-gate-only"],
                     [(1, gate_batch, "timing gate (1 chunk)")]
                     + [(1, b, n) for (_w, b, n) in LADDER if b < gate_batch],
                     upstream=["cpu_prep"])
    gate_lines = [l for l in Path(STAGES["timing_gate"]["log"]).read_text().splitlines()
                  if "timing gate:" in l]
    TIMING_GATE = gate_lines[-1].split("timing gate:", 1)[1].strip() if gate_lines else None
    print("timing gate reported by the codebase:", TIMING_GATE)
    m = re.search(r"'peak_vram_gib': ([0-9.]+)", TIMING_GATE or "")
    if m and float(m.group(1)) > 0:              # re-plan with the MEASURED GPU memory
        PLAN, LADDER = plan_resources(measured_vram_gib=float(m.group(1)))
    h = re.search(r"'est_total_gpu_hours': ([0-9.]+)", TIMING_GATE or "")
    if h:
        w0 = LADDER[0][0]
        print(f"estimated GPU solve: ~{float(h.group(1)):.1f} h with 1 worker/GPU; "
              f"with {w0} worker(s)/GPU roughly {float(h.group(1)) / w0:.1f}-"
              f"{float(h.group(1)) / max(1, w0) * 1.4:.1f} h (GPU sharing is not perfectly linear)")

print("resource plan:", json.dumps({k: v for k, v in PLAN.items() if k != "hardware"},
                                   indent=1, default=str))
(REPORT_DIR / "resource_plan.json").write_text(json.dumps(PLAN, indent=2, default=str))

GPU_SOLVE_OUTPUTS = [CODE_PATHS[k] for k in (
    "duckdb_path", "product1_parquet", "product2_monthly_parquet",
    "product2_daily_parquet", "product3_parquet")]
if CONFIG["RUN_GPU_SOLVE"]:
    extra = ["--max-sweeps", str(CONFIG["MAX_SWEEPS"])] if CONFIG["MAX_SWEEPS"] else []
    run_engine_stage("gpu_solve", [ENTRYPOINTS["gpu_solve"]], LADDER, GPU_SOLVE_OUTPUTS,
                     upstream=["cpu_prep"], extra_args=extra)
else:
    print("GPU solve skipped by configuration")

## 12. Stage 4 — scientific validation (codebase `run_env2_validate.py`)

In [ ]:
if CONFIG["RUN_VALIDATE"]:
    if "LADDER" not in globals():
        PLAN, LADDER = plan_resources()
    run_engine_stage("validate", [ENTRYPOINTS["validate"]], LADDER,
                     [CODE_PATHS["diagnostics_json"]], upstream=["gpu_solve"])
else:
    print("validation skipped by configuration")
(REPORT_DIR / "resource_attempts.json").write_text(json.dumps(RESOURCE_LOG, indent=2))
if any(r["result"] == "ok" and "smaller chunks" in r["note"] for r in RESOURCE_LOG):
    print("NOTE: a stage completed only with smaller chunks; see resource_attempts.json")
if CONFIG["RESUME_CHUNKS"] and STAGES.get("validate", {}).get("returncode") == 0:
    shutil.rmtree(RESUME_DIR, ignore_errors=True)     # all solves finished
MONITOR.stop()

## 13. Collect and verify the outputs

Checks that every product the codebase is expected to write exists and is readable
(Parquet footers and schemas, DuckDB tables and row counts, `diagnostics.json`), that the
`/kaggle/working` quota is respected, and smoke-tests the codebase's web API against the
produced database. Metrics shown are **read from the codebase's outputs**, never recomputed.

In [ ]:
import pyarrow.parquet as pq
import duckdb

INVENTORY, OUT_PROBLEMS = {}, []
for key in ("product1_parquet", "product2_monthly_parquet", "product2_daily_parquet",
            "product3_parquet", "pass_calibrated_parquet", "pyromes_parquet"):
    p = Path(CODE_PATHS[key])
    if not p.exists():
        OUT_PROBLEMS.append(f"missing {p.name}")
        continue
    try:
        meta = pq.ParquetFile(p).metadata
        INVENTORY[p.name] = {"bytes": p.stat().st_size, "rows": meta.num_rows,
                             "columns": meta.num_columns}
    except Exception as exc:
        OUT_PROBLEMS.append(f"unreadable {p.name}: {exc}")

db = Path(CODE_PATHS["duckdb_path"])
if db.exists():
    con = duckdb.connect(str(db), read_only=True)
    tables = [r[0] for r in con.execute(
        "SELECT table_name FROM information_schema.tables ORDER BY 1").fetchall()]
    INVENTORY[db.name] = {"bytes": db.stat().st_size,
                          "tables": {t: con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
                                     for t in tables}}
    con.close()
    for t in ("obs_harmonized_provenance", "state_reconstructed_monthly",
              "state_reconstructed_daily", "uncertainty_diagnostics", "hex_centers"):
        if t not in INVENTORY[db.name]["tables"]:
            OUT_PROBLEMS.append(f"duckdb lacks table {t}")
else:
    OUT_PROBLEMS.append("missing fire_harmonized.duckdb")

DIAG = None
dp = Path(CODE_PATHS["diagnostics_json"])
if dp.exists():
    DIAG = json.loads(dp.read_text())
    INVENTORY[dp.name] = {"bytes": dp.stat().st_size, "sections": sorted(DIAG)}
elif CONFIG["RUN_VALIDATE"]:
    OUT_PROBLEMS.append("missing diagnostics.json")

# web API smoke test (codebase FastAPI app, in-process, read-only DB)
WEB = None
if db.exists():
    _w = subprocess.run([sys.executable, "-c", (
        "import json, os; from fastapi.testclient import TestClient; "
        "from src.webapp import server as s; c = TestClient(s.app); "
        "r = c.get('/api/diagnostics'); "
        "print(json.dumps({'diagnostics_status': r.status_code}))")],
        cwd=CODE, env={**RUN_ENV, "FIRE_HARMONIZED_DB": str(db)},
        capture_output=True, text=True)
    WEB = _w.stdout.strip().splitlines()[-1] if _w.returncode == 0 else _w.stderr[-500:]

used = sum(p.stat().st_size for p in Path(CONFIG["OUTPUT_ROOT"]).rglob("*") if p.is_file())
print(json.dumps(INVENTORY, indent=1, default=str))
print(f"web API smoke test: {WEB}")
print(f"/kaggle/working usage: {used / 2**30:.2f} GiB (quota {CONFIG['WORKING_QUOTA_GB']} GiB)")
if used / 2**30 > CONFIG["WORKING_QUOTA_GB"]:
    OUT_PROBLEMS.append("outputs exceed the /kaggle/working quota")
if OUT_PROBLEMS:
    print("OUTPUT PROBLEMS:\n  " + "\n  ".join(OUT_PROBLEMS))
else:
    print("all expected outputs present and readable")

## 14. Provenance, run report and housekeeping

Writes `run_report/run_manifest.json` (package id, commit, sources, configuration,
environment, dependency versions, stage timings, test results, output inventory, resource
peaks), copies the package and data-provenance manifests next to it, and plots the resource
trace. Optionally removes the codebase's large intermediates from `/kaggle/working`.

In [ ]:
shutil.copy2(PKG_ROOT / MANIFEST_NAME, REPORT_DIR / MANIFEST_NAME)
prov = Path(COMPONENTS["staged_data"]) / "data_provenance_manifest.json"
if prov.exists():
    shutil.copy2(prov, REPORT_DIR / prov.name)

RUN_MANIFEST = {
    "run_id": RUN_ID,
    "finished_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "package_id": PACKAGE_ID,
    "spatial_subset": globals().get("SUBSET_INFO", "full"),
    "codebase": MANIFEST["codebase"],
    "acquisition": {k: MANIFEST.get("acquisition", {}).get(k) for k in
                    ("provenance_status_counts", "nrt_gap_fill", "warnings")},
    "config": CONFIG,
    "environment": {"python": platform.python_version(), "platform": platform.platform(),
                    "hardware": HW, "dependencies": DEP_VERSIONS},
    "stages": STAGES,
    "tests": {k: {kk: v.get(kk) for kk in ("passed", "summary", "seconds")}
              for k, v in TESTS.items()},
    "outputs": INVENTORY,
    "output_problems": OUT_PROBLEMS,
    "web_api_smoke": WEB,
    "resource_peaks": MONITOR.peaks,
}
(REPORT_DIR / "run_manifest.json").write_text(json.dumps(RUN_MANIFEST, indent=2, default=str))

try:
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    rs = pd.read_csv(REPORT_DIR / "resources.csv", parse_dates=["utc"])
    if len(rs):
        fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        cpu = rs.drop_duplicates("utc")
        ax[0].plot(cpu["utc"], cpu["cpu_pct"], label="CPU %")
        ax[0].plot(cpu["utc"], cpu["ram_used_gib"], label="RAM GiB")
        ax[0].legend(loc="upper left")
        g = rs.dropna(subset=["gpu"])
        for i, sub in g.groupby("gpu"):
            ax[1].plot(sub["utc"], sub["gpu_mem_used_gib"], label=f"GPU {int(i)} mem GiB")
        if len(g):
            ax[1].legend(loc="upper left")
        ax[1].set_xlabel("UTC")
        fig.tight_layout()
        fig.savefig(REPORT_DIR / "resources.png", dpi=110)
        plt.close(fig)
except Exception as exc:
    print("resource plot skipped:", exc)

if CONFIG["DELETE_INTERMEDIATES"] and not OUT_PROBLEMS:
    for key in ("pass_obs_parquet", "provenance_base_parquet"):
        _p = Path(CODE_PATHS[key])          # provenance_base is a per-year directory
        shutil.rmtree(_p, ignore_errors=True) if _p.is_dir() else _p.unlink(missing_ok=True)
    shutil.rmtree(Path(CODE_PATHS["state_checkpoint_pt"] + ".chunks"), ignore_errors=True)
print("run report:", REPORT_DIR)

## 15. Summary

In [ ]:
print(f"package     {PACKAGE_ID}")
print(f"subset      {globals().get('SUBSET_INFO', 'full')}")
print(f"codebase    {MANIFEST['codebase']['repo_url']} @ {MANIFEST['codebase']['commit'][:12]}")
print(f"GPUs        {[g['name'] for g in HW['gpus']] or 'none'}")
for name, rec in STAGES.items():
    state = "skipped (already done)" if rec.get("skipped") else f"exit {rec['returncode']}"
    print(f"stage       {name:<16} {state:<24} {rec.get('seconds', 0):>8}s")
for name, t in TESTS.items():
    print(f"tests       {name:<16} {t.get('summary', '')}")
if DIAG:
    sb = DIAG.get("structural_breaks", {})
    ab = DIAG.get("ablation_2019_2021", {})
    print("codebase diagnostics (from diagnostics.json):")
    for k in ("chow_raw_2012", "chow_harm_2012"):
        if k in sb:
            print(f"  {k:<18} F={sb[k].get('F'):.3g}  p={sb[k].get('p'):.3g}")
    if ab:
        print(f"  ablation 2019-21   coverage_90={ab.get('coverage_90'):.3f} "
              f"log_rmse={ab.get('log_rmse'):.3f} vs raw MODIS "
              f"{ab.get('raw_baseline', {}).get('log_rmse', float('nan')):.3f}")
print(f"outputs     {OUTPUT_DIR}")
print(f"report      {REPORT_DIR / 'run_manifest.json'}")
if OUT_PROBLEMS:
    raise RuntimeError("run finished with output problems: " + "; ".join(OUT_PROBLEMS))